##Cell 1 — Install + check GPU

In [ ]:
# Cell 1 — Install + check GPU + reproducibility setup

from google.colab import files
uploaded = files.upload()  # Upload raft.jsonl (and optionally a pinned requirements file later)

# Install core packages (current run)
!pip -q install unsloth trl datasets accelerate
# Optional: avoids some audio dependency conflicts/warnings
!pip -q uninstall -y torchaudio

# -----------------------------
# Reproducibility: set seeds
# -----------------------------
import os, random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(SEED)
np.random.seed(SEED)

# Torch is imported after install
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Make some operations more deterministic (best effort)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("SEED:", SEED)

# -----------------------------
# Reproducibility: record versions
# -----------------------------
!python -V
!pip show torch transformers trl unsloth datasets accelerate | tee colab_versions.txt
!pip freeze | sort > colab_pip_freeze.txt

print("✅ Saved version logs: colab_versions.txt, colab_pip_freeze.txt")


##Cell 2 — Load + format dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="raft.jsonl", split="train")
print("rows:", len(dataset))
print("cols:", dataset.column_names)
print("sample:", dataset[0])

def split_oracle_and_distractors(raw_input: str):
    text = raw_input.strip()

    oracle = text
    distractor_parts = []

    if "Distractors:" in text:
        oracle, distractors_text = text.split("Distractors:", 1)
        oracle = oracle.replace("Context (oracle):", "").strip()
        distractor_parts = [p.strip() for p in distractors_text.split("\n\n---\n\n") if p.strip()]
    else:
        oracle = text.replace("Context (oracle):", "").strip()

    return oracle, distractor_parts

def format_example(ex):
    oracle, distractors = split_oracle_and_distractors(ex["input"])

    source_blocks = [f"[S1] pdf=oracle_doc chunk_id=oracle_chunk\n{oracle}"]

    for i, d in enumerate(distractors[:3], start=2):
        source_blocks.append(f"[S{i}] pdf=distractor_doc chunk_id=distractor_chunk_{i-1}\n{d}")

    sources_text = "\n\n---\n\n".join(source_blocks)

    answer = ex["output"].strip()
    if "[S1]" not in answer:
        answer = answer + " [S1]"

    return {
        "text": (
            "You are a medical document QA assistant.\n"
            "RULES:\n"
            "- Use ONLY the SOURCES below.\n"
            "- If the answer is not clearly supported by the sources, say: "
            "\"I don't know based on the provided documents.\"\n"
            "- In your answer, cite sources like [S1], [S2] next to the claims they support.\n"
            "- Keep the answer concise and factual.\n\n"
            "QUESTION:\n"
            + ex["instruction"].strip() + "\n\n"
            "SOURCES:\n"
            + sources_text + "\n\n"
            "ANSWER:\n"
            + answer
            + "\n<END_ANSWER>"
        )
    }

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(dataset[0]["text"][:1500])




##Cell 3 — Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel

print("Using SEED =", SEED)

max_seq_length = 2048
dtype = None
load_in_4bit = True  # saves VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# If your installed unsloth version supports random_state, keep it.
# If it errors, remove random_state=SEED and rerun this cell.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    random_state=SEED,   # <-- optional but helpful if supported
)

##Cell 4 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling, set_seed

# Set Transformers seed too (important)
set_seed(SEED)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch = 16
        warmup_steps=10,
        max_steps=20,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        save_steps=100,

        # Reproducibility / cleaner runs
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0,
        report_to="none",   # avoids wandb prompt
    ),
)

train_result = trainer.train()
print("✅ Training finished")
print(train_result)


##Cell 5 — Quick inference test (sanity check)

In [ ]:
# Cell 5 — Real diabetes sanity check using project-style prompt

FastLanguageModel.for_inference(model)

prompt = """You are a medical document QA assistant.
RULES:
- Use ONLY the SOURCES below.
- If the answer is not clearly supported by the sources, say: "I don't know based on the provided documents."
- In your answer, cite sources like [S1], [S2] next to the claims they support.
- Keep the answer concise and factual.

QUESTION:
In NICE NG28, what is the first-line drug treatment recommended for adults with type 2 diabetes?

SOURCES:
[S1] pdf=NICE NG28-type-2-diabetes-in-adults-management-pdf-1837338615493 chunk_id=NICE NG28-type-2-diabetes-in-adults-management-pdf-1837338615493_chunk_0043_33a69aa1680a255d1e9a
For adults with type 2 diabetes and chronic kidney disease, follow recommendations on SGLT2 inhibitors in the section on chronic kidney disease in this guideline.

- 1.7.3 Offer standard-release metformin as first-line drug treatment to adults with type 2 diabetes. [2015]
- 1.7.4 Assess the person's cardiovascular status and risk to determine whether they have chronic heart failure or established atherosclerotic cardiovascular disease or are at high risk of developing cardiovascular disease.

ANSWER:
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

out = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False,
    repetition_penalty=1.15,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

prompt_len = inputs["input_ids"].shape[1]
new_tokens = out[0][prompt_len:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

for stop_marker in ["<END_ANSWER>", "\n---", "\n[S2]", "\n[S3]", "\n[S4]"]:
    if stop_marker in answer:
        answer = answer.split(stop_marker, 1)[0]

print(answer.strip())

##Cell 6 — Save adapter (LoRA)

In [ ]:
model.save_pretrained("raft_lora_adapter")
tokenizer.save_pretrained("raft_lora_adapter")

# Add reproducibility metadata file
import json
from pathlib import Path
import platform

meta = {
    "base_model": "unsloth/Qwen2.5-0.5B-Instruct",
    "seed": SEED,
    "max_seq_length": max_seq_length,
    "load_in_4bit": load_in_4bit,
    "lora": {
        "r": 16,
        "alpha": 16,
        "dropout": 0.0,
        "target_modules": [
            "q_proj","k_proj","v_proj","o_proj",
            "gate_proj","up_proj","down_proj",
        ],
    },
    "training": {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "effective_batch_size": 16,
        "warmup_steps": 10,
        "max_steps": 20,
        "learning_rate": 2e-4,
        "fp16": True,
    },
    "dataset": {
        "rows": len(dataset),
        "text_field": "text",
        "source_file_uploaded": "raft.jsonl",
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
}

Path("raft_lora_adapter/train_run_meta.json").write_text(
    json.dumps(meta, indent=2), encoding="utf-8"
)

print("✅ Saved metadata: raft_lora_adapter/train_run_meta.json")


##Cell 7 — OPTIONAL: merge + convert to GGUF + quantize

In [ ]:
# Merge LoRA into a full 16-bit model (optional, needed for GGUF conversion)
model.save_pretrained_merged("merged_model_16bit", tokenizer, save_method="merged_16bit")

!rm -rf /content/llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

# Pin llama.cpp to the exact working commit (reproducible)
LLAMA_CPP_COMMIT = "244641955f6146f7e8474afff7772d427593a534"
!cd /content/llama.cpp && git checkout {LLAMA_CPP_COMMIT}

!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model_16bit \
  --outfile /content/qwen2.5-0.5b-raft-f16.gguf --outtype f16

!ls -lh /content/qwen2.5-0.5b-raft-f16.gguf

from google.colab import files
files.download("/content/qwen2.5-0.5b-raft-f16.gguf")

# Build quantizer + quantize to Q8_0
!rm -rf /content/llama.cpp/build
!cmake -S /content/llama.cpp -B /content/llama.cpp/build \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLAMA_BUILD_TESTS=OFF \
  -DLLAMA_BUILD_EXAMPLES=OFF

!cmake --build /content/llama.cpp/build --target llama-quantize -j 1

!/content/llama.cpp/build/bin/llama-quantize \
  /content/qwen2.5-0.5b-raft-f16.gguf \
  /content/qwen2.5-0.5b-raft-q8_0.gguf \
  q8_0

!ls -lh /content/qwen2.5-0.5b-raft-q8_0.gguf
files.download("/content/qwen2.5-0.5b-raft-q8_0.gguf")
